# Project Milestone Two

**Data Preparation and Model Exploration**

**Note: No late assignments accepted, we need the time to grade them!**

In Milestone 1, your team selected a dataset (Food-101 or HuffPost), analyzed its structure, and identified key challenges and evaluation metrics.
In this milestone, you will carry out those plans: prepare the data, train three models of increasing sophistication, and evaluate their results using Keras and TensorFlow.
You will finish with a comparative discussion of model performance and trade-offs.


### Submission Guidelines

* Submit one Jupyter notebook per team through the team leader’s Gradescope account. **Include all team members names at the top of the notebook.** 
* Include all code, plots, and answers inline below.
* Ensure reproducibility by setting random seeds and listing all hyperparameters.
* Document any AI tools used, as required by the CDS policy.


## Problem 1 – Data Preparation and Splits (20 pts)

### Goals

Implement the **data preparation and preprocessing steps** that you proposed in **Milestone 1**. You’ll clean, normalize, and split your data so that it’s ready for modeling and reproducible fine-tuning.

### Steps to Follow

1. **Load your chosen dataset**

   * Use `datasets.load_dataset()` from **Hugging Face** to load **Food-101** or **HuffPost**.
   * Display basic information (e.g., number of samples, feature names, example entries).

2. **Apply cleaning and normalization**

   * **Images:**

     * Ensure all images are in RGB format.
     * Resize or crop to a consistent shape (e.g., `224 × 224`).
     * Drop or fix any corrupted files.
   * **Text:**

     * Concatenate headline + summary (for HuffPost).
     * Strip whitespace, convert to lowercase if appropriate, and remove empty samples.
     * Optionally remove duplicates or extremely short entries.

3. **Standardize or tokenize the inputs**

   * **Images:**

     * Normalize pixel values (e.g., divide by 255.0).
     * Define a minimal augmentation pipeline (e.g., random flip, crop, or rotation).
   * **Text:**

     * Create a tokenizer or `TextVectorization` layer.
     * Set a target `max_length` based on your analysis from Milestone 1 (e.g., 95th percentile).
     * Apply padding/truncation and build tensors for input + labels.

4. **Handle dataset-specific challenges**

   * If you identified **class imbalance**, compute label counts and, if needed, create a dictionary of `class_weights`.
   * If you noted **length or size variance**, verify that your truncation or resizing works as intended.
   * If you planned **noise filtering**, include the cleaning step and briefly explain your criteria (e.g., remove items with missing text or unreadable images).

5. **Create reproducible splits**

   * Split your cleaned dataset into **train**, **validation**, and **test** subsets (e.g., 80 / 10 / 10).
   * Use a fixed random seed for reproducibility (`random_seed = 42`).
   * Use **stratified splits**  (e.g., with `train_test_split` and `stratify = labels`).
   * Display the size of each subset.

6. **Document your pipeline**

   * Summarize your preprocessing steps clearly in Markdown or code comments.
   * Save or display a few representative examples after preprocessing to confirm the transformations are correct.




In [1]:
#load hf token
import os
token = os.getenv("HUGGINGFACE_TOKEN")
#other imports
from datasets import load_dataset, DatasetDict
from datasets.features import ClassLabel
import pandas as pd

#### Step 1: Load dataset from Hugging Face

In [2]:
# Your code here; add as many cells as you need but make it clear what the structure is. 
#1. load dataset:
# --- Hugging Face Datasets
URL = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"
dataset = load_dataset("json", data_files=URL, split="train")

#convert to pandas
huffpost = dataset.to_pandas()

#display basic info
print("Number of samples:", len(huffpost))
print("Columns:", huffpost.columns)
huffpost.head(3)

Number of samples: 200853
Columns: Index(['category', 'headline', 'authors', 'link', 'short_description', 'date'], dtype='str')


,category,headline,authors,link,short_description,date
0,CRIME,There Were 2 Mass Shootings In Texas Last Week...,Melissa Jeltsen,https://www.huffingtonpost.com/entry/texas-ama...,She left her husband. He killed their children...,2018-05-26
1,ENTERTAINMENT,Will Smith Joins Diplo And Nicky Jam For The 2...,Andy McDonald,https://www.huffingtonpost.com/entry/will-smit...,Of course it has a song.,2018-05-26
2,ENTERTAINMENT,Hugh Grant Marries For The First Time At Age 57,Ron Dicker,https://www.huffingtonpost.com/entry/hugh-gran...,The actor and his longtime girlfriend Anna Ebe...,2018-05-26


#### Step 2: Combine headline and description into a single text field, normalize text and remove short and duplicate samples.

In [3]:
#2. cleaning and normalization
#combine headline and description
huffpost["text"] = huffpost["headline"].fillna("") + " [SEP] " + huffpost["short_description"].fillna("")
#normalize text
huffpost["text"] = huffpost["text"].str.lower().str.strip()
#remove empty/very short text data
huffpost = huffpost[huffpost["text"].str.len() > 20]
#remove duplicates
huffpost = huffpost.drop_duplicates(subset="text")
#print remaining samples
print("Remaining samples:", len(huffpost))

Remaining samples: 200185


In [4]:
#3. tokenization/standardization
#create tokenizer
from tqdm import tqdm
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

input_ids = []
attention_masks = []

#tokenize in batches so kernel doesnt crash
for text in tqdm(huffpost["text"]):
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    input_ids.append(encoded["input_ids"])
    attention_masks.append(encoded["attention_mask"])

huffpost["input_ids"] = input_ids
huffpost["attention_mask"] = attention_masks

100%|██████████| 200185/200185 [00:52<00:00, 3800.00it/s]


#### Step 4: Handle class imbalance with class weights, handle dataset-specific challenges, and check length variances

In [5]:
#4. handle dataset-specific challenges
#class imbalance
label_counts = huffpost["category"].value_counts()
print(label_counts)

category
POLITICS          32707
WELLNESS          17821
ENTERTAINMENT     16048
TRAVEL             9877
STYLE & BEAUTY     9511
PARENTING          8649
HEALTHY LIVING     6667
QUEER VOICES       6310
FOOD & DRINK       6225
BUSINESS           5928
COMEDY             5094
SPORTS             4880
BLACK VOICES       4522
HOME & LIVING      4161
PARENTS            3896
THE WORLDPOST      3664
WEDDINGS           3651
IMPACT             3456
DIVORCE            3422
CRIME              3400
WOMEN              3400
MEDIA              2811
WEIRD NEWS         2665
GREEN              2613
WORLDPOST          2571
RELIGION           2546
STYLE              2246
SCIENCE            2177
WORLD NEWS         2176
TASTE              2092
TECH               2033
MONEY              1706
ARTS               1505
GOOD NEWS          1397
FIFTY              1394
ARTS & CULTURE     1338
ENVIRONMENT        1322
COLLEGE            1143
LATINO VOICES      1129
CULTURE & ARTS     1029
EDUCATION          1003
Name: c

In [6]:
#4. handle dataset-specific challenges
#create class weights
total = len(huffpost)
class_weights = {label: total / count for label, count in label_counts.items()}
class_weights

{'POLITICS': 6.120555232824778,
 'WELLNESS': 11.233095785870603,
 'ENTERTAINMENT': 12.474140079760717,
 'TRAVEL': 20.267793864533765,
 'STYLE & BEAUTY': 21.047734202502365,
 'PARENTING': 23.145450341079894,
 'HEALTHY LIVING': 30.026248687565623,
 'QUEER VOICES': 31.725039619651348,
 'FOOD & DRINK': 32.15823293172691,
 'BUSINESS': 33.769399460188936,
 'COMEDY': 39.29819395367099,
 'SPORTS': 41.021516393442624,
 'BLACK VOICES': 44.26912870411322,
 'HOME & LIVING': 48.1098293679404,
 'PARENTS': 51.382186858316224,
 'THE WORLDPOST': 54.6356441048035,
 'WEDDINGS': 54.83018351136675,
 'IMPACT': 57.92390046296296,
 'DIVORCE': 58.49941554646406,
 'CRIME': 58.877941176470586,
 'WOMEN': 58.877941176470586,
 'MEDIA': 71.21487015297048,
 'WEIRD NEWS': 75.11632270168856,
 'GREEN': 76.61117489475699,
 'WORLDPOST': 77.86269933877868,
 'RELIGION': 78.62725844461902,
 'STYLE': 89.12956366874444,
 'SCIENCE': 91.95452457510335,
 'WORLD NEWS': 91.99678308823529,
 'TASTE': 95.69072657743786,
 'TECH': 98.46

In [7]:
#4. handle dataset-specific challenges
#check length variance 
text_lengths = huffpost["text"].str.len()
print("Max length:", text_lengths.max())
print("Average length:", text_lengths.mean())

Max length: 1493
Average length: 179.34842770437345


#### Step 7: Split dataset into train/validation/test (80/10/10) with stratification

In [8]:
#5. reproducible train/val/test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
huffpost["label"] = le.fit_transform(huffpost["category"])
num_classes = len(le.classes_)
print("Number of classes:", num_classes)

train_df, temp_df = train_test_split(
    huffpost,
    test_size=0.2,
    stratify=huffpost["category"],
    random_state=42)

#second split
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["category"],
    random_state=42
)

#display sizes
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Number of classes: 41
Train size: 160148
Validation size: 20018
Test size: 20019


#### 6. Documentation and Verification

In [9]:
#show processed examples
train_df[["text", "category"]].head()

,text,category
55876,russia's chances of competing in rio 2016 trac...,SPORTS
22734,find your breath while reporting in the age of...,MEDIA
106013,immigration fight is psychological warfare [se...,POLITICS
76893,the best jobs for work-life balance [sep] ever...,HEALTHY LIVING
165499,chronic stress raises diabetes risk [sep] the ...,WELLNESS


### Graded Questions (5 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **Data Loading and Cleaning:**
   Describe how you loaded your dataset and the key cleaning steps you implemented (e.g., handling missing data, normalizing formats, or removing duplicates).



1.1. **Your answer here:**

I loaded the HuffPost dataset using load_dataset() from Hugging Face and converted it into a pandas DataFrame for easier manipulation. I combined the headline and short description into a single text field, normalized it by lowercasing and stripping whitespace, and removed missing or very short entries to reduce noise. I also dropped duplicate samples based on the text field to improve data quality.

2. **Preprocessing and Standardization:**
   Summarize your preprocessing pipeline. Include any normalization, tokenization, resizing, or augmentation steps, and explain why each was necessary for your dataset.
  

1.2. **Your answer here:**

I tokenized the text using a BERT tokenizer (bert-base-uncased) with padding and truncation applied to a fixed max_length of 128. This ensured consistent input size for the model and handled variability in text length. Normalization (lowercasing) improved consistency, while truncation prevented excessively long inputs from impacting performance.

3. **Train/Validation/Test Splits:**
   Explain how you divided your data into subsets, including the split ratios, random seed, and any stratification or leakage checks you used to verify correctness.


1.3. **Your answer here:**

I split the dataset into 80% training, 10% validation, and 10% test sets using train_test_split with a fixed random seed of 42 for reproducibility. Stratified sampling based on the category labels ensured that each subset preserved the original class distribution. This approach helps prevent data leakage and ensures fair model evaluation.

4. **Class Distribution and Balance:**
   Report your label counts and describe any class imbalances you observed. If applicable, explain how you addressed them (e.g., weighting, oversampling, or data augmentation).


1.4. **Your answer here:**

I examined label counts using value_counts() and observed class imbalance across categories, with some classes having significantly more samples than others. To address this, I computed class weights using inverse frequency, which can be used during training to reduce bias toward majority classes. This helps the model learn more balanced representations across all categories.

## Problem 2 – Baseline Model (20 pts)

### Goal

Build and train a **simple, fully functional baseline model** to establish a reference level of performance for your dataset.
This baseline will help you evaluate whether later architectures and fine-tuning steps actually improve results.


### Steps to Follow

1. **Construct a baseline model**

   * **Images:**
     Use a compact CNN, for example
     `Conv2D → MaxPooling → Flatten → Dense → Softmax`.
   * **Text:**
     Use a small embedding-based classifier such as
     `Embedding → GlobalAveragePooling → Dense → Softmax`.
   * Keep the model small enough to train in minutes on Colab.

2. **Compile the model**

   * Optimizer: `Adam` or `AdamW`.
   * Loss: `categorical_crossentropy` (for multi-class).
   * Metrics: at least `accuracy`; add `F1` if appropriate.

3. **Train and validate**

   * Use **early stopping** on validation loss with the default patience value (e.g., 5 epochs).
   * Record number of epochs trained and total runtime.

4. **Visualize results**

   * Plot **training vs. validation accuracy and loss**.
   * Carefully observe: does the model underfit, overfit, or generalize reasonably?

5. **Report baseline performance**

   * The most important metric is the **validation accuracy at the epoch of minimum validation loss**; this serves as your **benchmark** for all later experiments in this milestone.
   * Evaluate on the **test set** and record final metrics.

#### 2.1 Construct Baseline Model

In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = tf.keras.Sequential([
    layers.Embedding(input_dim=30522, output_dim=128, input_length=128),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

I0000 00:00:1775676594.319131   73896 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775676611.750344   73896 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


#### 2.2 Compile Baseline Model

In [11]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

E0000 00:00:1775676616.229065   73896 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


#### 2.3 Train & Early Stopping

In [ ]:
#create inputs & labels
#import numpy as np

#train on sample of df so it doesnt crash
#train_df = train_df.sample(25000, random_state=42)
#val_df = val_df.sample(10000, random_state=42)
#test_df = test_df.sample(10000, random_state=42)

#train_inputs = np.array(train_df["input_ids"].tolist())
#train_labels = np.array(train_df["label"].tolist())

#val_inputs = np.array(val_df["input_ids"].tolist())
#val_labels = np.array(val_df["label"].tolist())

#test_inputs = np.array(test_df["input_ids"].tolist())
#test_labels = np.array(test_df["label"].tolist())

: 

In [ ]:
#train w/early stopping
from tensorflow.keras.callbacks import EarlyStopping

#use subset and tf dataset to avoid crashing
import numpy as np

#reduce size
train_df = train_df.sample(30000, random_state=42)
val_df = val_df.sample(10000, random_state=42)

#convert to list
train_inputs = np.array(train_df["input_ids"].tolist())
train_labels = np.array(train_df["label"].tolist())

val_inputs = np.array(val_df["input_ids"].tolist())
val_labels = np.array(val_df["label"].tolist())

# tf dataset
train_dataset = tf.data.Dataset.from_tensor_slices((train_inputs, train_labels)).batch(32)
val_dataset = tf.data.Dataset.from_tensor_slices((val_inputs, val_labels)).batch(32)


early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True)

# train
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[early_stop])

#history = model.fit(
    #train_inputs,
    #train_labels,
    #validation_data=(val_inputs, val_labels),
    #epochs=15,
    #batch_size=32,
    #callbacks=[early_stop])

Epoch 1/20


W0000 00:00:1775676617.873073   73896 cpu_allocator_impl.cc:82] Allocation of 30720000 exceeds 10% of free system memory.
W0000 00:00:1775676618.001124   73896 cpu_allocator_impl.cc:82] Allocation of 30720000 exceeds 10% of free system memory.


324/938 ━━━━━━━━━━━━━━━━━━━━ 24s 40ms/step - accuracy: 0.1583 - loss: 3.3974

#### 2.4 Visualize Results

In [ ]:
import matplotlib.pyplot as plt

#plot accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train', 'Validation'])
plt.show()

In [ ]:
#plot loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train', 'Validation'])
plt.show()

#### 2.5 Evaluate on Test Set & Report Performance

In [ ]:
test_loss, test_acc = model.evaluate(test_inputs, test_labels)
print("Test accuracy:", test_acc)

### Graded Questions (5 pts each)

1. **Model Architecture:**
   Describe your baseline model and justify why this structure suits your dataset.

2.1. **Your answer here:**



2. **Training Behavior:**
   Summarize the model’s training and validation curves. What trends did you observe?

2.2. **Your answer here:**



  3. **Baseline Metrics:**
   Report validation and test metrics. What does this performance tell you about dataset difficulty?

2.3. **Your answer here:**



  4. **Reflection:**
   What are the main limitations of your baseline? Which specific improvements (depth, regularization, pretraining) would you try next?
  

2.4. **Your answer here:**



## Problem 3 – Custom (Original) Model (20 pts)

### Goal

Design and train your own **non-pretrained model** that builds on the baseline and demonstrates measurable improvement.
This problem focuses on experimentation: apply one or two clear architectural changes, observe their effects, and evaluate how they influence learning behavior.


### Steps to Follow

1. **Modify or extend your baseline architecture**

   * Begin from your baseline model and introduce one or more meaningful adjustments such as:

     * Adding **dropout** or **batch normalization** for regularization.
     * Increasing **depth** (extra convolutional or dense layers).
     * Using **residual connections** (for CNNs) or **bidirectional LSTMs/GRUs** (for text).
     * Trying alternative activations like `ReLU`, `LeakyReLU`, or `GELU`.
   * Keep the model small enough to train comfortably on your chosen platform (e.g., Colab)

2. **Observe what specific limitations you want to address**

   * Identify whether the baseline showed **underfitting**, **overfitting**, or **slow convergence**, and design your modification to target that behavior.
   * Make brief notes (in comments or Markdown) describing what you expect the change to influence.

3. **Train and evaluate under the same conditions**

   * Use the **same data splits**, **random seed**, and **metrics** as in Problem 2.
   * Apply **early stopping** on validation loss.
   * Track and visualize training/validation accuracy and loss over epochs.

4. **Compare outcomes to the baseline**

   * Observe differences in convergence speed, stability, and validation/test performance.
   * Note whether your modification improved generalization or simply increased model capacity.

### Graded Questions (5 pts each)

1. **Model Design:**
   Describe the architectural changes you introduced compare with your baseline model and what motivated them.

3.1. **Your answer here:**



2. **Training Results:**
   Present key validation and test metrics. Did your modifications improve performance?

3.2. **Your answer here:**



3. **Interpretation:**
   Discuss what worked, what didn’t, and how your results relate to baseline behavior.

3.3. **Your answer here:**



4. **Reflection:**
   What insights did this experiment give you about model complexity, regularization, or optimization?

3.4. **Your answer here:**



## Problem 4 – Pretrained Model (Transfer Learning) (20 pts)

### Goal

Apply **transfer learning** to see how pretrained knowledge improves accuracy, convergence speed, and generalization.
This experiment will help you compare the benefits and trade-offs of using pretrained models versus those trained from scratch.


### Steps to Follow

1. **Select a pretrained architecture**

   * **Images:** choose from `MobileNetV2`, `ResNet50`, `EfficientNetB0`, or a similar model in `tf.keras.applications`.
   * **Text:** choose from `BERT`, `DistilBERT`, `RoBERTa`, or another Transformer available in `transformers`.

2. **Adapt the model for your dataset**

   * Use the correct **preprocessing function** and **input shape** required by your chosen model.
   * Replace the top layer with your own **classification head** (e.g., `Dense(num_classes, activation='softmax')`).

3. **Apply transfer learning**

   * Choose an appropriate **training strategy** for your pretrained model. Options include:

     * **Freezing** the pretrained base and training only a new classification head.
     * **Partially fine-tuning** selected upper layers of the base model.
     * **Full fine-tuning** (all layers trainable) with a reduced learning rate.
   * Adjust your learning rate schedule to match your strategy (e.g., smaller LR for fine-tuning).
   * Observe how your chosen approach affects **validation loss**, **training time**, and **model stability**.

4. **Train and evaluate under consistent conditions**

   * Use the same **splits**, **metrics**, and **evaluation protocol** as in earlier problems.
   * Record training duration, validation/test performance, and any resource constraints (GPU memory, runtime).

5. **Compare and analyze**

   * Observe how transfer learning changes both **performance** and **efficiency** relative to your baseline and custom models.
   * Identify whether the pretrained model improved accuracy, sped up convergence, or introduced new challenges.


### Graded Questions (5 pts each)

1. **Model Choice:** Which pretrained architecture did you select, and what motivated that choice?

4.1. **Your answer here:**



2. **Fine-Tuning Plan:** Describe your fine-tuning strategy and why you chose it. 

4.2. **Your answer here:**



3. **Performance:** Report key metrics and compare them with your baseline and custom models.

4.3. **Your answer here:**



4. **Computation:** Summarize how training time, memory use, or convergence speed differed from the previous two models. 

4.4. **Your answer here:**



## Problem 5 – Comparative Evaluation and Discussion (20 pts)

### Goal

Compare your **baseline**, **custom**, and **pretrained** models to evaluate how design choices affected performance, efficiency, and generalization.
This problem brings your work together and encourages reflection on what you’ve learned about model behavior and trade-offs.

**Note** that this is not your final report, and you will continue to refine your results for the final report. 

### Steps to Follow

1. **Compile key results**

   * Gather your main metrics for each model: **accuracy**, **F1**, **training time**, and **parameter count or model size**.
   * Ensure all numbers come from the same evaluation protocol and test set.

2. **Visualize the comparison**

   * Present results in a **single, well-organized chart or table**.
   * Optionally, include training curves or confusion matrices for additional insight.

3. **Analyze comparative performance**

   * Observe which model performed best by your chosen metric(s).
   * Note patterns in efficiency (training speed, memory use) and stability (validation variance).

4. **Inspect model behavior**

   * Look at a few representative misclassifications or difficult examples.
   * Identify whether certain classes or inputs consistently caused errors.

5. **Plan forward improvements**

   * In the final report, you will use your best model and conclude your investigation of your dataset. Based on your observations, decide on a model and next steps for refining your approach in the final project (e.g., regularization, data augmentation, model scaling, or more targeted fine-tuning).

### Graded Questions (4 pts each)

1. **Summary Table and Performance Analysis:** Present a clear quantitative comparison of all three models. Which model achieved the best overall results, and what factors contributed to its success?

5.1. **Your answer here:**



2. **Trade-Offs:** Discuss how complexity, accuracy, and efficiency balanced across your models.

5.2. **Your answer here:**



3. **Error Patterns:** Describe the types of examples or classes that remained challenging for all models.

5.3. **Your answer here:**



4. **Next Steps:** Based on these findings, decide on a model to go forward with and outline your plan for improving that model. 


5.4 **Your answer here:**



### Final Question: Describe what use you made of generative AI tools in preparing this Milestone. 

**AI Question: Your answer here:**